# Inferencia RAD-ALERT
Este notebook toma el dataset limpio (`data_cleaned.xlsx`), prepara el texto concatenando y normalizando las columnas relevantes,
ejecuta la inferencia con el modelo RAD-ALERT, y exporta el conjunto de datos de los reportes "Críticos" para entrenamiento.


In [8]:
pip install transformers torch


  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.7 MB 3.4 MB/s eta 0:00:04
   -------- ------------------------------- 2.4/10.7 MB 7.5 MB/s eta 0:00:02
   --------------- ------------------------ 4.2/10.7 MB 7.9 MB/s eta 0:00:01
   ----------------------- ---------------- 6.3/10.7 MB 8.6 MB/s eta 0:00:01
   ------------------------------- -------- 8.4/10.7 MB 9.0 MB/s eta 0:00:01
   -------------------------------------- - 10.2/10.7 MB 9.0 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 8.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/616.3 kB ? eta -:--:--
   ---------------------------------------- 616.3/616.3 kB 7.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ------------------- -----------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import re
import unidecode
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import warnings
warnings.filterwarnings('ignore')


c:\Users\nsp1324\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# Cargar el dataset limpio
df = pd.read_excel('../data/data_cleaned.xlsx')
print(f"Total de registros originales: {len(df)}")

# Función para limpiar y normalizar texto al estilo RAD-ALERT
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower() # minúsculas
    text = unidecode.unidecode(text) # sin acentos
    text = re.sub(r'[^a-z0-9\s]', ' ', text) # eliminar caracteres especiales
    text = re.sub(r'\s+', ' ', text).strip() # espacios extra
    return text

# Concatenar las columnas de interés
cols_texto = ['Datos Clínicos', 'Hallazgos', 'Opinión']
df['texto_informe'] = df[cols_texto].fillna('').agg(' '.join, axis=1)

# Aplicar normalización
df['texto_normalizado'] = df['texto_informe'].apply(normalize_text)

df[['texto_informe', 'texto_normalizado']].head()


Total de registros originales: 3977


,texto_informe,texto_normalizado
0,cefalea intensa con signos de alarma. descarta...,cefalea intensa con signos de alarma descartar...
1,tce. cefalea y emesis persistente. signos de ...,tce cefalea y emesis persistente signos de san...
2,cefalea con signos de alarma. dx mielomeningoc...,cefalea con signos de alarma dx mielomeningoce...
3,trauma facial y trauma cráneo encefálico. sig...,trauma facial y trauma craneo encefalico signo...
4,alteracion del estado de consiencia. surcos y...,alteracion del estado de consiencia surcos y e...


In [19]:
# Configurar modelo RAD-ALERT
# IMPORTANTE: Reemplaza 'ruta/al/modelo/rad-alert' con el path real local o el nombre en HuggingFace.
model_name = "../../RAD-ALERT/model"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    print(f"Modelo cargado correctamente en {device}.")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    print("Asegúrate de haber instalado los requerimientos de RAD-ALERT e indicar la ruta correcta del modelo.")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2673.77it/s]

Modelo cargado correctamente en cpu.


In [ ]:
# Función para predecir
def predict_critical(text):
    if not text.strip():
        return "No crítico", 0.0
        
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        # Ajusta el índice (1 o 0) según qué clase represente 'Crítico' en tu modelo
        prob_critico = probs[0][1].item() 
        
    pred = "Crítico" if prob_critico >= 0.5 else "No crítico"
    return pred, prob_critico

# Aplicar a todo el dataset
# Descomenta las siguientes líneas cuando el modelo esté configurado:
print("Iniciando inferencia...")
df['rad_prediccion'], df['rad_score'] = zip(*df['texto_normalizado'].apply(predict_critical))
print(df['rad_prediccion'].value_counts())


Iniciando inferencia...


In [18]:
# Filtrar y Exportar
# Descomenta esto después de ejecutar la inferencia exitosamente.

df_criticos = df[df['rad_prediccion'] == 'Crítico'].copy()
print(f"Total de registros Críticos identificados por RAD-ALERT: {len(df_criticos)}")

# Guardar el dataset final como rad_criticos.xlsx
df_criticos.to_excel('../data/rad_criticos.xlsx', index=False)
print("Exportado exitosamente a '../data/rad_criticos.xlsx'")


Total de registros Críticos identificados por RAD-ALERT: 952
Exportado exitosamente a '../data/rad_criticos.xlsx'
